# Spectral bandwidth distribution: N = 1000, lambda = N^(3/2), M = 1000

Extend [the 100-repetition experiment](spectral_bandwidth_distribution_n1000_lambda_n3_2_m100.ipynb) to $M=1000$, with $N=1000$, $\lambda_N=N^{3/2}$, and $n=200$. Preserve the geometric grid on [0.01, 2], Gaussian samples, seed `20260828`, FFT interval [-12,12), 8,192 spatial points, and shared stabilized Perron solver.

Reuse the completed first 100 repetitions after checking their settings and sample hash. Reuse the exact ISE and Q-hat selections from the earlier 1,000-sample run only after matching its complete sample hash; otherwise calculate them directly. The remaining 900 spectral selections are computed on the subsequent samples in the same sequence. Each repetition retains Perron residual and spectral-gap diagnostics.

Automatic PDF export is enabled in setup. This notebook has separate data, image, and PDF directories.

Sources: [100-repetition experiment](spectral_bandwidth_distribution_n1000_lambda_n3_2_m100.ipynb); [1,000-sample comparison selectors](figure_3_5_bandwidth_distribution_n1000.ipynb); [shared Perron solver](kde_tools/perron.py).


In [1]:
from __future__ import annotations
import csv
import hashlib
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Code" / "Code_kde" / "kde_tools").is_dir())
code_directory = project_root / "Code" / "Code_kde"
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))
os.environ.setdefault("MPLCONFIGDIR", str(project_root / ".matplotlib-cache"))
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, MultipleLocator
import numpy as np
from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta, gaussian_mise_oracle_bandwidth
from kde_tools.density import exact_oracle_ise_values
from kde_tools.fourier import beta2_divergence_matrix_fft
from kde_tools.perron import PERRON_SOLVER_VERSION, perron_peak_index, perron_eigenpair_from_divergence
from kde_tools.pdf_export import enable_automatic_pdf_saving
from kde_tools.similarities import beta2_divergence_matrix

notebook_name = "spectral_bandwidth_distribution_n1000_lambda_n3_2_m1000"
pdf_output_directory = enable_automatic_pdf_saving(notebook_name, output_root=code_directory / "outputs" / "pdfs")
run_directory = code_directory / "outputs" / "data" / notebook_name
image_directory = code_directory / "outputs" / "images" / notebook_name
for directory in (run_directory, image_directory):
    directory.mkdir(parents=True, exist_ok=True)
seed = 20260818


In [2]:
beta2_n = 200
beta2_N = 1000
beta2_M = 1000

beta2_h_min = 0.01
beta2_h_max = 2.0
beta2_lambda = beta2_N ** (3.0 / 2.0)
beta2_seed = seed + 10
beta2_half_width = 12.0
beta2_spatial_points = 8192
beta2_validate_fft = True

if beta2_n < 1 or beta2_M < 1:
    raise ValueError("beta2_n and beta2_M must be positive.")
if beta2_N < 2:
    raise ValueError("beta2_N must be at least 2.")
if not 0.0 < beta2_h_min < beta2_h_max:
    raise ValueError("Require 0 < beta2_h_min < beta2_h_max.")
if beta2_lambda <= 0.0:
    raise ValueError("beta2_lambda must be positive.")
if beta2_half_width <= 0.0:
    raise ValueError("beta2_half_width must be positive.")
if beta2_spatial_points < 2:
    raise ValueError("beta2_spatial_points must be at least 2.")

beta2_h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=beta2_h_min,
    h_max=beta2_h_max,
    beta=2.0,
    grid_size=beta2_N,
)

print(                                                                                              ### REPORTING
    f"beta=2, n={beta2_n}, N={beta2_N}, M={beta2_M}, "                                              ### REPORTING
    f"lambda={beta2_lambda:g}"                                                                      ### REPORTING
)                                                                                                   ### REPORTING

beta=2, n=200, N=1000, M=1000, lambda=31622.8


In [3]:
def exact_beta2_qhat_values(
    samples: np.ndarray,
    h_grid: np.ndarray,
) -> np.ndarray:
    """Evaluate Q-hat_(n,2) exactly on a bandwidth grid."""
    squared_differences = (
        samples[:, None] - samples[None, :]
    ) ** 2
    values = np.empty(len(h_grid), dtype=float)
    for index, bandwidth in enumerate(h_grid):
        q = 2.0 * bandwidth**2
        overlap = (
            np.exp(-squared_differences / (2.0 * q))
            / np.sqrt(2.0 * np.pi * q)
        )
        second_q_factor = (
            3.0 / (4.0 * q**2)
            - 3.0 * squared_differences / (2.0 * q**3)
            + squared_differences**2 / (4.0 * q**4)
        )
        values[index] = (
            2.0
            * bandwidth**4
            * np.mean(overlap * second_q_factor)
        )
    return values


In [4]:
validation_rng = np.random.default_rng(beta2_seed)
validation_samples = validation_rng.normal(size=min(beta2_n, 100))
validation_indices = np.unique(np.linspace(0, beta2_N - 1, min(beta2_N, 20), dtype=int))
validation_h_grid = beta2_h_grid[validation_indices]
validation_exact = beta2_divergence_matrix(validation_samples, validation_h_grid)
validation_fft = beta2_divergence_matrix_fft(validation_samples, validation_h_grid, beta2_half_width, beta2_spatial_points)
validation_absolute_error = float(np.max(np.abs(validation_fft - validation_exact)))
validation_relative_error = validation_absolute_error / float(np.max(validation_exact))
assert validation_relative_error < 2e-5
print(f"FFT validation: absolute error={validation_absolute_error:.3e}, relative error={validation_relative_error:.3e}")

settings = {
    "n": beta2_n, "N": beta2_N, "M": beta2_M,
    "h_min": beta2_h_min, "h_max": beta2_h_max, "lambda": beta2_lambda,
    "lambda_rule": "N**(3/2)", "seed": beta2_seed, "fft_half_width": beta2_half_width,
    "fft_spatial_points": beta2_spatial_points, "perron_solver": PERRON_SOLVER_VERSION,
}
(run_directory / "settings.json").write_text(json.dumps(settings, indent=2) + "\n")

# Draw all 1,000 samples in exactly the original sequential RNG order.
rng = np.random.default_rng(beta2_seed)
full_samples = np.array([rng.normal(size=beta2_n) for _ in range(beta2_M)])
validation = {
    "changed_experiment_settings": {"M": {"baseline": 100, "repeat": 1000}},
    "previous_experiment": "spectral_bandwidth_distribution_n1000_lambda_n3_2_m100",
    "unchanged_grid_size": beta2_N,
    "source_notebook": str(code_directory / "spectral_scale_selection_kde.ipynb"),
    "source_cells": ["beta2-configurable-parameters", "beta2-configurable-selections", "beta2-configurable-histograms"],
    "fft_maximum_absolute_error": validation_absolute_error,
    "fft_relative_error": validation_relative_error,
    "samples_sha256": hashlib.sha256(full_samples.tobytes()).hexdigest(),
    "workers": 4,
}
(run_directory / "validation.json").write_text(json.dumps(validation, indent=2) + "\n")


FFT validation: absolute error=2.176e-14, relative error=1.197e-13


In [5]:
checkpoint_path = run_directory / "checkpoint.npz"
selected_indices = np.full((beta2_M, 3), -1, dtype=int)
completed = np.zeros(beta2_M, dtype=bool)
solver_diagnostics = np.full((beta2_M, 3), np.nan)
settings_json = json.dumps(settings, sort_keys=True)
if checkpoint_path.exists():
    with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
        assert str(checkpoint["settings_json"].item()) == settings_json
        assert np.array_equal(checkpoint["h_grid"], beta2_h_grid)
        selected_indices[:] = checkpoint["selected_indices"]
        completed[:] = checkpoint["completed"]
        solver_diagnostics[:] = checkpoint["solver_diagnostics"]


# Reuse the first 100 completed spectral results after verifying inputs.
previous_directory = code_directory / "outputs/data/spectral_bandwidth_distribution_n1000_lambda_n3_2_m100"
previous_path = previous_directory / "checkpoint.npz"
if not checkpoint_path.exists() and previous_path.exists():
    with np.load(previous_path, allow_pickle=False) as previous:
        previous_settings = json.loads(str(previous["settings_json"].item()))
        previous_M = int(previous_settings["M"])
        assert {key: value for key, value in previous_settings.items() if key != "M"} == {key: value for key, value in settings.items() if key != "M"}
        assert previous_M <= beta2_M
        assert np.array_equal(previous["h_grid"], beta2_h_grid)
        previous_validation = json.loads((previous_directory / "validation.json").read_text())
        assert hashlib.sha256(full_samples[:previous_M].tobytes()).hexdigest() == previous_validation["samples_sha256"]
        selected_indices[:previous_M] = previous["selected_indices"]
        completed[:previous_M] = previous["completed"]
        solver_diagnostics[:previous_M] = previous["solver_diagnostics"]
        validation["reused_previous_repetitions"] = int(previous["completed"].sum())
        validation["previous_sample_prefix_hash_verified"] = True

# ISE and Q-hat are independent of lambda; retain the verified saved values.
control_indices = None
control_directory = code_directory / "outputs/data/figure_3_5_bandwidth_distribution_n1000"
control_path = control_directory / "selections.csv"
if control_path.exists() and (control_directory / "validation.json").exists():
    control_validation = json.loads((control_directory / "validation.json").read_text())
    control_settings = json.loads((control_directory / "settings.json").read_text())
    assert control_validation["samples_sha256"] == validation["samples_sha256"]
    for key in ("n", "N", "M", "h_min", "h_max", "seed", "fft_half_width", "fft_spatial_points", "perron_solver"):
        assert control_settings[key] == settings[key]
    with control_path.open() as stream:
        control_rows = list(csv.DictReader(stream))
    assert len(control_rows) == beta2_M
    control_indices = np.array([[int(row["ise_grid_index"]), int(row["qhat_grid_index"])] for row in control_rows])
    assert np.all((control_indices >= 0) & (control_indices < beta2_N))
    assert np.array_equal(selected_indices[completed, 1:], control_indices[completed])
    validation["cached_ise_qhat_complete_sample_hash_verified"] = True
    validation["cached_ise_qhat_source"] = str(control_path)

def save_checkpoint():
    temporary = checkpoint_path.with_suffix(".npz.tmp")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, settings_json=settings_json, h_grid=beta2_h_grid,
                            selected_indices=selected_indices, completed=completed, solver_diagnostics=solver_diagnostics)
    temporary.replace(checkpoint_path)

def select_one(repetition):
    current_samples = full_samples[repetition]
    divergence = beta2_divergence_matrix_fft(current_samples, beta2_h_grid, beta2_half_width, beta2_spatial_points)
    eigenpair = perron_eigenpair_from_divergence(divergence, beta2_lambda)
    spectral = perron_peak_index(eigenpair.vector)
    if control_indices is None:
        ise = int(np.argmin(exact_oracle_ise_values(current_samples, beta2_h_grid, sigma2=1.0)))
        qhat = int(np.argmin(exact_beta2_qhat_values(current_samples, beta2_h_grid)))
    else:
        ise, qhat = map(int, control_indices[repetition])
    return repetition, (spectral, ise, qhat), (eigenpair.relative_residual, eigenpair.relative_gap, eigenpair.matvecs)

started = time.perf_counter()
print(f"Starting at {int(completed.sum())}/{beta2_M} completed repetitions", flush=True)
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(select_one, int(r)) for r in np.flatnonzero(~completed)]
    for future in as_completed(futures):
        repetition, indices, diagnostics = future.result()
        selected_indices[repetition] = indices
        solver_diagnostics[repetition] = diagnostics
        completed[repetition] = True
        total = int(completed.sum())
        if total % 10 == 0 or total == beta2_M:
            save_checkpoint()
        if total % 100 == 0 or total == beta2_M:
            print(f"Completed {total}/{beta2_M}; elapsed {time.perf_counter()-started:.1f}s", flush=True)
assert np.all(completed)
assert np.all((0 <= selected_indices) & (selected_indices < beta2_N))
counts = np.stack([np.bincount(selected_indices[:, column], minlength=beta2_N) for column in range(3)])
assert np.all(counts.sum(axis=1) == beta2_M)
validation["completed_repetitions"] = int(completed.sum())
validation["all_three_count_totals"] = counts.sum(axis=1).tolist()
validation["elapsed_seconds"] = time.perf_counter() - started
validation["maximum_perron_relative_residual"] = float(solver_diagnostics[:, 0].max())
validation["minimum_perron_relative_gap"] = float(solver_diagnostics[:, 1].min())
assert np.all(np.isfinite(solver_diagnostics))
(run_directory / "validation.json").write_text(json.dumps(validation, indent=2) + "\n")


Starting at 100/1000 completed repetitions
Completed 200/1000; elapsed 17.7s
Completed 300/1000; elapsed 33.4s
Completed 400/1000; elapsed 48.1s
Completed 500/1000; elapsed 63.8s
Completed 600/1000; elapsed 78.3s
Completed 700/1000; elapsed 92.4s
Completed 800/1000; elapsed 109.9s
Completed 900/1000; elapsed 125.9s
Completed 1000/1000; elapsed 149.1s


In [6]:
labels = ["spectral selections", "ISE-optimal grid bandwidths", "Q-hat minimizers"]
with (run_directory / "selections.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["repetition", "spectral_grid_index", "spectral_h", "ise_grid_index", "ise_h", "qhat_grid_index", "qhat_h"])
    for repetition, indices in enumerate(selected_indices):
        writer.writerow([repetition, int(indices[0]), beta2_h_grid[indices[0]], int(indices[1]), beta2_h_grid[indices[1]], int(indices[2]), beta2_h_grid[indices[2]]])
with (run_directory / "frequencies.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["grid_index", "h", "spectral_count", "ise_count", "qhat_count"])
    for index, h in enumerate(beta2_h_grid):
        writer.writerow([index, h, *counts[:, index].tolist()])
summary = []
for column, label in enumerate(labels):
    values = beta2_h_grid[selected_indices[:, column]]
    summary.append({"selector": label, "mean_h": float(values.mean()), "median_h": float(np.median(values)),
                    "modal_h": float(beta2_h_grid[np.argmax(counts[column])]), "repetitions": beta2_M})
with (run_directory / "summary.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=summary[0].keys())
    writer.writeheader()
    writer.writerows(summary)
print(json.dumps(summary, indent=2))

with (run_directory / "solver_diagnostics.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["repetition", "relative_residual", "relative_gap", "matvecs"])
    writer.writerows([r, *row] for r, row in enumerate(solver_diagnostics))

# ISE and Q-hat do not use lambda; compare them on the same saved samples.
baseline_path = code_directory / "outputs/data/figure_3_5_bandwidth_distribution_n1000/selections.csv"
if baseline_path.exists():
    with baseline_path.open() as stream:
        baseline_rows = list(csv.DictReader(stream))[:beta2_M]
    if len(baseline_rows) == beta2_M:
        expected = np.array([[int(row["ise_grid_index"]), int(row["qhat_grid_index"])] for row in baseline_rows])
        assert np.array_equal(selected_indices[:, 1:], expected)
        validation["ise_and_qhat_match_all_baseline_samples"] = True
        validation["baseline_selections"] = str(baseline_path)
(run_directory / "validation.json").write_text(json.dumps(validation, indent=2) + "\n")

# The extension must preserve every result from the previous 100-sample run.
if previous_path.exists():
    with np.load(previous_path, allow_pickle=False) as previous:
        previous_M = len(previous["completed"])
        assert np.array_equal(selected_indices[:previous_M], previous["selected_indices"])
        assert np.array_equal(solver_diagnostics[:previous_M], previous["solver_diagnostics"])
    validation["all_first_100_selections_and_diagnostics_unchanged"] = True
(run_directory / "validation.json").write_text(json.dumps(validation, indent=2) + "\n")


[
  {
    "selector": "spectral selections",
    "mean_h": 0.26343161829990125,
    "median_h": 0.2651306870999343,
    "modal_h": 0.25819232639474754,
    "repetitions": 1000
  },
  {
    "selector": "ISE-optimal grid bandwidths",
    "mean_h": 0.37482963716043605,
    "median_h": 0.36835508009332074,
    "modal_h": 0.3644684980252875,
    "repetitions": 1000
  },
  {
    "selector": "Q-hat minimizers",
    "mean_h": 0.2657929442882146,
    "median_h": 0.26867042210662695,
    "modal_h": 0.2932400808459329,
    "repetitions": 1000
  }
]


In [7]:
# Preserve the three original discrete distributions and the MISE reference.
colors = ["#C65D00", "#9467BD", "#66C2A5"]
opacities = [0.70, 0.70, 0.55]
fig, ax = plt.subplots(figsize=(12.0, 6.32))
fig.subplots_adjust(left=0.085, right=0.98, top=0.88, bottom=0.25)
grid_positions = np.arange(beta2_N)
BIN_GRID_STEPS = 5  # Combine five consecutive candidate bandwidths per bin.
bin_starts = np.arange(0, beta2_N, BIN_GRID_STEPS)
bin_stops = np.minimum(bin_starts + BIN_GRID_STEPS, beta2_N)
bin_centers = (bin_starts + bin_stops - 1) / 2.0
bin_widths = bin_stops - bin_starts
binned_counts = np.add.reduceat(counts, bin_starts, axis=1)
assert np.array_equal(binned_counts.sum(axis=1), counts.sum(axis=1))
assert np.all(binned_counts.sum(axis=1) == beta2_M)
for label, color, alpha, values in zip(labels, colors, opacities, binned_counts):
    nonzero = np.flatnonzero(values)
    ax.bar(bin_centers[nonzero], values[nonzero] / beta2_M,
           width=0.94 * bin_widths[nonzero], color=color, alpha=alpha,
           edgecolor="white", linewidth=0.3, label=label)
beta2_mise_bandwidth = gaussian_mise_oracle_bandwidth(beta2_n)
reference = np.interp(np.log(beta2_mise_bandwidth), np.log(beta2_h_grid), grid_positions)
ax.axvline(reference, color="#1f77b4", linestyle=(0, (5, 5)), linewidth=1.5,
           label=f"Gaussian MISE optimum: h={beta2_mise_bandwidth:.4g}")
ticks = np.unique(np.linspace(0, beta2_N - 1, 7, dtype=int))
ax.set_xticks(ticks, [f"{beta2_h_grid[i]:.3g}" for i in ticks])
ax.set_xlim(-0.5, beta2_N - 0.5)
ax.set_ylim(0, 1.05 * float(binned_counts.max()) / beta2_M)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.yaxis.set_major_locator(MultipleLocator(0.01))
ax.set_xlabel("selected bandwidth h", fontsize=12, labelpad=12, color="#304967")
ax.set_ylabel("relative frequency", fontsize=12, labelpad=12, color="#304967")
ax.tick_params(axis="both", labelsize=10, colors="#304967", length=0)
ax.grid(axis="y", color="#E5ECF6", linewidth=0.6)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#E5ECF6")
fig.suptitle("Bandwidth distributions: 5 consecutive grid points per bin\n"
             + r"$N=1000$, $n=200$, $M=1000$, $\lambda_N=N^{3/2}$",
             y=0.965, fontsize=14, color="#304967")
handles, legend_labels = ax.get_legend_handles_labels()
order = [legend_labels.index(label) for label in labels] + [legend_labels.index(f"Gaussian MISE optimum: h={beta2_mise_bandwidth:.4g}")]
fig.legend([handles[i] for i in order], [legend_labels[i] for i in order], loc="lower center",
           bbox_to_anchor=(0.52, 0.035), ncol=2, frameon=False, fontsize=10, labelcolor="#304967")
png_path = image_directory / "spectral_distribution_N1000_lambda_N3_2_M1000_bins5.png"
fig.savefig(png_path, dpi=180, facecolor="white")
plt.show()  # The shared automatic exporter saves this same figure as PDF.
plt.close(fig)
print(f"Figure: {png_path}")
print(f"PDF directory: {pdf_output_directory}")

with (run_directory / "frequencies_bins5.csv").open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["bin_index", "first_grid_index", "last_grid_index", "h_first", "h_last", "spectral_count", "ise_count", "qhat_count"])
    for b, (first, stop) in enumerate(zip(bin_starts, bin_stops)):
        writer.writerow([b, int(first), int(stop - 1), beta2_h_grid[first], beta2_h_grid[stop - 1], *binned_counts[:, b].tolist()])
print(f"Binning: {BIN_GRID_STEPS} grid points per bin; {len(bin_starts)} bins per distribution")


Saved PDF: /Users/iljasukhanov/MASTERARBEIT/Code/Code_kde/outputs/pdfs/spectral_bandwidth_distribution_n1000_lambda_n3_2_m1000/001_bandwidth_distributions_5_consecutive_grid_points_per_bin_n_1000_n_200_m_1000_la.pdf
Figure: /Users/iljasukhanov/MASTERARBEIT/Code/Code_kde/outputs/images/spectral_bandwidth_distribution_n1000_lambda_n3_2_m1000/spectral_distribution_N1000_lambda_N3_2_M1000_bins5.png
PDF directory: /Users/iljasukhanov/MASTERARBEIT/Code/Code_kde/outputs/pdfs/spectral_bandwidth_distribution_n1000_lambda_n3_2_m1000
Binning: 5 grid points per bin; 200 bins per distribution
